In [ ]:
import pandas as pd
import os

RAW_PATH = "../data/raw"

dim_file = os.path.join(RAW_PATH, "DimTables.xlsx")
internet_file = os.path.join(RAW_PATH, "FactInternetSales.xlsx")
reseller_file = os.path.join(RAW_PATH, "FactResellerSales.xlsx")

print("Checking raw dataset files...\n")

for file in [dim_file, internet_file, reseller_file]:
    print(f"{os.path.basename(file)} : {'Found' if os.path.exists(file) else 'NOT FOUND'}")

In [ ]:
dim_excel = pd.ExcelFile(dim_file)

print("Dimension table sheets:")
for sheet in dim_excel.sheet_names:
    print(" -", sheet)

fact_internet = pd.read_excel(internet_file)
fact_reseller = pd.read_excel(reseller_file)

print("\nFact Internet Sales shape:", fact_internet.shape)
print("Fact Reseller Sales shape:", fact_reseller.shape)

In [ ]:

dim_customer = pd.read_excel(dim_file, sheet_name="DimCustomer")
dim_employee = pd.read_excel(dim_file, sheet_name="DimEmployee")
dim_geography = pd.read_excel(dim_file, sheet_name="DimGeography")
dim_product = pd.read_excel(dim_file, sheet_name="DimProduct")
dim_reseller = pd.read_excel(dim_file, sheet_name="DimReseller")
dim_sales_territory = pd.read_excel(dim_file, sheet_name="DimSalesTerritory")

print("Dimension table shapes:\n")

print("DimCustomer:", dim_customer.shape)
print("DimEmployee:", dim_employee.shape)
print("DimGeography:", dim_geography.shape)
print("DimProduct:", dim_product.shape)
print("DimReseller:", dim_reseller.shape)
print("DimSalesTerritory:", dim_sales_territory.shape)

In [ ]:

datasets = {
    "DimCustomer": dim_customer,
    "DimEmployee": dim_employee,
    "DimGeography": dim_geography,
    "DimProduct": dim_product,
    "DimReseller": dim_reseller,
    "DimSalesTerritory": dim_sales_territory,
    "FactInternetSales": fact_internet,
    "FactResellerSales": fact_reseller
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(df.dtypes)

In [ ]:

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(df.isnull().sum())

In [ ]:

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name}: {duplicate_count} duplicate rows")

In [ ]:

key_columns = {
    "DimCustomer": "CustomerKey",
    "DimEmployee": "EmployeeKey",
    "DimGeography": "GeographyKey",
    "DimProduct": "ProductKey",
    "DimReseller": "ResellerKey",
    "DimSalesTerritory": "SalesTerritoryKey"
}

for name, key in key_columns.items():
    df = datasets[name]
    duplicate_count = df[key].duplicated().sum()
    print(f"{name} - {key}: {duplicate_count} duplicate values")

In [ ]:

fact_reseller[fact_reseller.duplicated(keep=False)]

In [ ]:

for name, df in datasets.items():
    blank_rows = df.isna().all(axis=1).sum()
    print(f"{name}: {blank_rows} completely blank rows")

In [ ]:

for name, df in datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")
    print("-" * len(name))

    if missing.empty:
        print("No missing values")
    else:
        print(missing)

In [ ]:

fact_reseller[fact_reseller.isna().any(axis=1)]

In [ ]:

problem_rows = fact_reseller[fact_reseller.isna().any(axis=1)]

print("Total problematic rows:", len(problem_rows))
print("Completely blank rows:", problem_rows.isna().all(axis=1).sum())
print(
    "Rows with SalesAmount but no ProductKey:",
    problem_rows["SalesAmount"].notna().sum()
)
print(
    "Rows with TotalProductCost but no ProductKey:",
    problem_rows["TotalProductCost"].notna().sum()
)

In [ ]:

problem_rows = fact_reseller[fact_reseller["ProductKey"].isna()]

print("Problematic rows with sales values:")
print(
    problem_rows[
        ["TotalProductCost", "SalesAmount"]
    ].describe()
)

print("\nNormal transaction rows:")
print(
    fact_reseller[
        fact_reseller["ProductKey"].notna()
    ][["TotalProductCost", "SalesAmount"]].describe()
)

In [ ]:

for name, df in {
    "FactInternetSales": fact_internet,
    "FactResellerSales": fact_reseller
}.items():
    
    print(f"\n{name}")
    print("-" * len(name))
    
    for column in ["OrderDate", "DueDate", "ShipDate"]:
        converted = pd.to_datetime(df[column], errors="coerce")
        invalid = converted.isna().sum()
        
        print(f"{column}: {invalid} invalid or missing dates")

In [ ]:

for name, df in {
    "FactInternetSales": fact_internet,
    "FactResellerSales": fact_reseller
}.items():

    print(f"\n{name}")
    print("-" * len(name))

    for column in ["SalesAmount", "TotalProductCost", "DiscountAmount", "Freight"]:
        negative = (df[column] < 0).sum()
        zero = (df[column] == 0).sum()

        print(f"{column}: Negative = {negative}, Zero = {zero}")

In [ ]:

checks = {
    "InternetSales - CustomerKey": (
        fact_internet["CustomerKey"],
        dim_customer["CustomerKey"]
    ),
    "InternetSales - ProductKey": (
        fact_internet["ProductKey"],
        dim_product["ProductKey"]
    ),
    "InternetSales - SalesTerritoryKey": (
        fact_internet["SalesTerritoryKey"],
        dim_sales_territory["SalesTerritoryKey"]
    ),
    "ResellerSales - ProductKey": (
        fact_reseller["ProductKey"],
        dim_product["ProductKey"]
    ),
    "ResellerSales - ResellerKey": (
        fact_reseller["ResellerKey"],
        dim_reseller["ResellerKey"]
    ),
    "ResellerSales - EmployeeKey": (
        fact_reseller["EmployeeKey"],
        dim_employee["EmployeeKey"]
    ),
    "ResellerSales - SalesTerritoryKey": (
        fact_reseller["SalesTerritoryKey"],
        dim_sales_territory["SalesTerritoryKey"]
    )
}

for name, (fact_key, dim_key) in checks.items():
    invalid = (~fact_key.isna()) & (~fact_key.isin(dim_key))
    print(f"{name}: {invalid.sum()} invalid references")

In [ ]:

categorical_columns = {
    "DimCustomer": ["MaritalStatus", "Gender", "Education", "Occupation"],
    "DimProduct": ["Color", "Model", "Subcategory", "Category"],
    "DimReseller": ["BusinessType"],
    "DimSalesTerritory": ["SalesTerritoryRegion", "SalesTerritoryCountry", "SalesTerritoryGroup"]
}

for name, columns in categorical_columns.items():
    print(f"\n{name}")
    print("-" * len(name))

    df = datasets[name]

    for column in columns:
        print(f"\n{column}:")
        print(df[column].dropna().unique())

In [ ]:

numeric_checks = {
    "DimCustomer": ["YearlyIncome", "HouseOwnerFlag"],
    "DimProduct": ["ProductKey"],
    "DimGeography": ["GeographyKey"],
    "DimReseller": ["ResellerKey"],
    "DimSalesTerritory": ["SalesTerritoryKey"]
}

for name, columns in numeric_checks.items():
    print(f"\n{name}")
    print("-" * len(name))
    
    for column in columns:
        print(f"\n{column}:")
        print("Min:", datasets[name][column].min())
        print("Max:", datasets[name][column].max())

In [ ]:

date_checks = {
    "DimCustomer": ["BirthDate", "FirstPurchaseDate"],
    "FactInternetSales": ["OrderDate", "DueDate", "ShipDate"],
    "FactResellerSales": ["OrderDate", "DueDate", "ShipDate"]
}

for name, columns in date_checks.items():
    print(f"\n{name}")
    print("-" * len(name))

    df = datasets[name]

    for column in columns:
        dates = pd.to_datetime(df[column], errors="coerce")
        print(f"{column}: {dates.min()} → {dates.max()}")

In [ ]:

print("FactInternetSales columns:")
print(fact_internet.columns.tolist())

print("\nFactResellerSales columns:")
print(fact_reseller.columns.tolist())

In [ ]:

cleaned_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

In [ ]:

for name, df in cleaned_datasets.items():
    cleaned_datasets[name] = df.dropna(how="all").copy()

print("Completely blank rows removed from working datasets.")

In [ ]:

cleaned_datasets["FactResellerSales"] = cleaned_datasets["FactResellerSales"][
    cleaned_datasets["FactResellerSales"]["ProductKey"].notna()
].copy()

print(
    "FactResellerSales rows after removing non-transaction rows:",
    len(cleaned_datasets["FactResellerSales"])
)

In [ ]:

cleaned_datasets["DimProduct"]["Color"] = (
    cleaned_datasets["DimProduct"]["Color"].fillna("Unknown")
)

print(
    "Missing Color values after treatment:",
    cleaned_datasets["DimProduct"]["Color"].isna().sum()
)

In [ ]:

cleaned_datasets["DimEmployee"]["ParentEmployeeKey"] = (
    cleaned_datasets["DimEmployee"]["ParentEmployeeKey"].fillna(0)
)

print(
    "Missing ParentEmployeeKey values after treatment:",
    cleaned_datasets["DimEmployee"]["ParentEmployeeKey"].isna().sum()
)

In [ ]:

date_columns = ["OrderDate", "DueDate", "ShipDate"]

for name in ["FactInternetSales", "FactResellerSales"]:
    for column in date_columns:
        cleaned_datasets[name][column] = pd.to_datetime(
            cleaned_datasets[name][column],
            errors="coerce"
        )

print("Fact table date columns converted to datetime format.")

In [ ]:

for name, df in cleaned_datasets.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")
    print("-" * len(name))

    if missing.empty:
        print("No missing values")
    else:
        print(missing)

In [ ]:

cleaned_datasets["DimSalesTerritory"][
    cleaned_datasets["DimSalesTerritory"]["SalesTerritoryRegion"].isna()
]

In [ ]:

territory_columns = [
    "SalesTerritoryRegion",
    "SalesTerritoryCountry",
    "SalesTerritoryGroup"
]

cleaned_datasets["DimSalesTerritory"][territory_columns] = (
    cleaned_datasets["DimSalesTerritory"][territory_columns].fillna("Unknown")
)

print(
    "Remaining missing values in DimSalesTerritory:",
    cleaned_datasets["DimSalesTerritory"][territory_columns].isna().sum().sum()
)

In [ ]:

print("Final Dataset Validation")
print("=" * 30)

for name, df in cleaned_datasets.items():
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.duplicated().sum())
    print("Completely blank rows:", df.isna().all(axis=1).sum())

In [ ]:

PROCESSED_PATH = "../data/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

for name, df in cleaned_datasets.items():
    file_path = os.path.join(PROCESSED_PATH, f"{name}.csv")
    df.to_csv(file_path, index=False)

print("Cleaned datasets saved successfully.")

In [ ]:

for name in cleaned_datasets.keys():
    file_path = os.path.join(PROCESSED_PATH, f"{name}.csv")
    
    check_df = pd.read_csv(file_path)
    
    print(f"{name}.csv: {check_df.shape}")